# Dolos — real talking-head video generation (Colab GPU)

**Run-all notebook.** `Runtime → Run all` turns a text script into a talking-head video on a free Colab GPU, then prints a download link. No manual fixing — every SadTalker-vs-modern-Colab compatibility patch is already baked into step 2.

It mirrors exactly what the Dolos worker does (`apps/worker/app/pipeline/openmontage.py`):

1. **Piper TTS** — script → `narration.wav` (free/local)
2. **SadTalker** — face photo + that audio → talking-head `.mp4` (GPU)

**Before you run:** `Runtime → Change runtime type → Hardware accelerator: GPU` (free T4 is fine).

- To change what it says, edit **`SCRIPT`** in step 3.
- To use your own face, upload a front-facing photo and set **`AVATAR`** in step 4.

> **Why step 6 uploads to a link instead of showing the video inline:** Colab's inline player and `files.download()` rely on a JavaScript bridge that breaks with a *"Could not load the JavaScript files / third-party cookies"* error. Step 6 sidesteps that entirely by uploading the result and printing a plain URL you open in a new tab.

## 1. Confirm we actually have a GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Install SadTalker + its model checkpoints

Clones the repo, installs pinned deps, patches the `basicsr` import that newer torchvision breaks, and downloads the ~2GB of model weights. Takes a few minutes.

In [ ]:
import os

if not os.path.isdir('/content/SadTalker'):
    !git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker

%cd /content/SadTalker

# SadTalker was built for Python 3.8-3.10 + numpy 1.x. Colab now runs Python
# 3.12 + numpy 2.x, so its original pins (numpy==1.23.5, scipy==1.10.1, etc.)
# have NO 3.12 wheels and silently fail to install. We pin numpy==1.26.4 -
# the newest 1.x, which HAS 3.12 wheels and still exposes the old attributes
# (e.g. VisibleDeprecationWarning) SadTalker relies on - and leave the rest
# unpinned so pip resolves 3.12-compatible versions.
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q librosa numba resampy pydub scipy kornia yacs pyyaml joblib \
    scikit-image basicsr facexlib gfpgan face_alignment imageio imageio-ffmpeg av safetensors
!pip install -q "numpy==1.26.4"   # re-pin last: some dep above pulls numpy 2.x back in

import pathlib, re

# Patch 1: basicsr imports `torchvision.transforms.functional_tensor`, removed
# in newer torchvision. Redirect it to the current module.
import torchvision
deg = pathlib.Path(torchvision.__file__).parent / 'transforms' / 'functional_tensor.py'
if not deg.exists():
    deg.write_text('from torchvision.transforms.functional import rgb_to_grayscale\n')
    print('patched torchvision.transforms.functional_tensor shim')

# Patch 2: SadTalker's source uses long-removed numpy aliases (np.float, np.int,
# ...). Rewrite them to the builtins across the repo (np.float64/np.int32/etc.
# are left untouched by the word-boundary match).
alias_pats = [(re.compile(r'\b(?:np|numpy)\.%s\b' % a), a)
              for a in ['float', 'int', 'bool', 'object', 'str', 'complex']]
patched = 0
for p in pathlib.Path('/content/SadTalker').rglob('*.py'):
    t0 = p.read_text(); t = t0
    for pat, a in alias_pats:
        t = pat.sub(a, t)
    if t != t0:
        p.write_text(t); patched += 1
print('patched numpy-alias files:', patched)

# Patch 3: align_img builds np.array([w0, h0, s, t[0], t[1]]) where t[0]/t[1]
# are 1-element arrays. Newer numpy rejects that ragged mix; extract scalars.
pp = pathlib.Path('/content/SadTalker/src/face3d/util/preprocess.py')
src = pp.read_text()
src, n = re.subn(
    r'np\.array\(\[\s*w0\s*,\s*h0\s*,\s*s\s*,\s*t\[0\]\s*,\s*t\[1\]\s*\]\)',
    'np.array([w0, h0, s, float(np.reshape(t[0], -1)[0]), float(np.reshape(t[1], -1)[0])])',
    src)
pp.write_text(src)
print('patched align_img trans_params:', n)

# Download the model weights (~2GB).
!bash scripts/download_models.sh
print('SadTalker ready')

## 3. Piper TTS — script → narration.wav

The same free/local TTS our worker uses. Edit `SCRIPT` to whatever you want the avatar to say.

In [ ]:
SCRIPT = "Hello! This is a test of the Dolos video generation pipeline running on a GPU."

!pip install -q piper-tts

# Download a Piper voice model (small, ~60MB).
!mkdir -p /content/piper_voices
!wget -q -O /content/piper_voices/en_US-lessac-medium.onnx \
    https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx
!wget -q -O /content/piper_voices/en_US-lessac-medium.onnx.json \
    https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json

import subprocess
AUDIO = '/content/narration.wav'
subprocess.run(
    ['piper', '--model', '/content/piper_voices/en_US-lessac-medium.onnx',
     '--output_file', AUDIO],
    input=SCRIPT, capture_output=True, text=True, check=True,
)

from IPython.display import Audio
print('narration.wav generated')
Audio(AUDIO)

## 4. Pick the avatar photo

Uses SadTalker's own bundled test portrait (`examples/source_image/full3.png`) — the exact same default our worker falls back to. To use your own face, upload a front-facing photo and set `AVATAR` to its path.

In [ ]:
AVATAR = '/content/SadTalker/examples/source_image/full3.png'

from IPython.display import Image
Image(AVATAR, width=256)

## 5. SadTalker — photo + audio → talking-head video

This is the actual generation. `--preprocess full --still` keeps the whole framed photo and animates just the face (matches our worker's intent). First run is slower; a short clip takes ~1-3 min on a T4.

In [ ]:
%cd /content/SadTalker
!python inference.py \
    --driven_audio /content/narration.wav \
    --source_image {AVATAR} \
    --result_dir /content/results \
    --preprocess full \
    --still

import glob
vids = sorted(glob.glob('/content/results/**/*.mp4', recursive=True), key=os.path.getmtime)
assert vids, 'No output video found — check the SadTalker log above for errors.'
OUTPUT = vids[-1]
print('generated:', OUTPUT)

## 6. Get the video

Uploads the result to a file host and prints a URL — **open that URL in a new browser tab** to watch or download it.

We do it this way because Colab's `files.download()` and inline `<video>` player depend on a JavaScript bridge that fails when third-party cookies are blocked. A plain-text URL always works.

In [ ]:
# Upload to catbox.moe (no expiry) and print a direct URL. Falls back to
# tmpfiles.org if catbox is unavailable.
url = !curl -s -F "reqtype=fileupload" -F "fileToUpload=@{OUTPUT}" https://catbox.moe/user/api.php
link = url[-1].strip() if url else ''

if link.startswith('http'):
    print('Your video is ready — open this URL in a new browser tab:\n')
    print('   ' + link)
else:
    print('catbox upload failed (%r), trying tmpfiles.org...\n' % (url,))
    alt = !curl -s -F "file=@{OUTPUT}" https://tmpfiles.org/api/v1/upload
    print(alt)
    print('\n^ take the URL from that JSON and insert /dl/ after the domain,')
    print('  e.g. https://tmpfiles.org/dl/12345/video.mp4')

## What this proves & what's next

You just generated a real talking-head video with the same `Piper → SadTalker` chain the Dolos worker runs. **The pipeline is validated.**

Colab can't be the production backend — it times out, and its ToS forbids running a server. To go live, run the Dolos Celery worker on a **persistent GPU** (RunPod / Vast / your own NVIDIA box) with:

```
WORKER_STUB_MODE=false
OPENMONTAGE_REPO_PATH=/opt/OpenMontage
SADTALKER_PATH=/opt/SadTalker
```

pointed at the same Redis / Postgres / storage as the API.

**Important for deploy — reproduce this exact environment.** SadTalker only ran here after the compatibility fixes baked into step 2: pinning `numpy==1.26.4`, the `torchvision.transforms.functional_tensor` shim, and the numpy source patches (removed aliases like `np.float`, plus the `align_img` ragged-array fix). The worker's GPU image must reproduce that setup — either bake these same fixes into the image, or build on a pinned **Python 3.10** image where SadTalker installs cleanly with no patches. `apps/worker/app/pipeline/openmontage.py` already invokes SadTalker exactly as this notebook does.